# Importing libraries

In [2]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
import time
import os

# Gemini
import google.generativeai as genai

# Utilities and metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Setting seeds

In [3]:
s1 = 4279876
s2 = 5522499
s3 = 4674627

seeds = [s1, s2, s3]

# Importing datasets

In [4]:
ds = load_dataset("cardiffnlp/tweet_eval", "hate")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2970 entries, 0 to 2969
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2970 non-null   object
 1   label   2970 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 46.5+ KB


# Gemini Configuration

In [5]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

safety_settings = [
    {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'threshold': 'BLOCK_NONE'},
    {'category': 'HARM_CATEGORY_HATE_SPEECH', 'threshold': 'BLOCK_NONE'},
    {'category': 'HARM_CATEGORY_HARASSMENT', 'threshold': 'BLOCK_NONE'},
    {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'threshold': 'BLOCK_NONE'}
]

generation_config = {
    'temperature': 0.0
}

model = genai.GenerativeModel('gemini-1.5-flash', safety_settings=safety_settings, generation_config=generation_config)

# Zero-Shot Tests

In [6]:
def classify(text):
    prompt = f'''

    Read the following text and determine if it contains or mentions hate speech. If it does, respond 1, if it does not, respond 0. Do not respond anything else.
    Text: "{text}"
    Response: 

    '''

    output = model.generate_content(prompt)
    return output.text

In [7]:
sample = test.groupby('label').head(500)

results = pd.DataFrame()
y_preds = []

for i, row in sample.iterrows():
    text = row['text']
    while True:
        try:
            print("Text:", text)
            print("True label:", row['label'])
            start = time.perf_counter()
            pred = classify(text)
            inference_time = time.perf_counter() - start
            print("Predicted label:", pred)

            columns = {
                "text": text,
                "true_label": row['label'],
                "predicted_label": pred,
                "inference_time": inference_time
            }

            results = pd.concat([results, pd.DataFrame(columns, index=[0])])
            break

        except:
            print("Api limit reached, sleeping for 10 seconds")
            time.sleep(10)
    


Text: @user , you are correct that Reid certainly is a weasel. Sadly, we've got our own weasels; @user Sen McConnell & @user .The corrupt Mueller investigation w/be STOPPED if those 3 did their jobs.#MAGA #KAG #POTUS #Trump #NEWS #VoteRed #NoDACA #USA
True label: 0
Predicted label: 1

Text: Whoever just unfollowed me you a bitch
True label: 1
Predicted label: 1 

Text: @user @user Those People Invaded Us!!! They DO NOT BELING HERE & HAVE NO RIGHTS! Its #AmericaFIRST! Open Your House To Them If Your That IGNORANT! & Yes Im A #Christian Too! #NODACA!
True label: 1
Predicted label: 1 

Text: stop JUDGING bitches by there cover, jus cuz she bad don't mean she's a catch shawdy could be a whore 👀 das opposite of a keeper
True label: 1
Predicted label: 1 

Text: how about i knock heads off and send them gift wrapped to your moms house you dumb raggedy bird bitch ass hoes
True label: 1
Predicted label: 1 

Text: @user @user @user Always #NoDACA.I AM BORN IN #USA AND #USA FIRST.
True label: 0
P

KeyboardInterrupt: 

# Process results